# Verify k66 Parser Registration

This notebook creates a new dynamic parser plugin at `src/parsers/k66`, registers it through the existing discovery system, and validates that it appears in `reports/index.html` and `parser_inventory.json`.

In [ ]:
from pathlib import Path
import json
import os

ROOT = Path("e:/PROJECTS/DronLibreFirmwaretools").resolve()
assert ROOT.exists(), "Workspace root not found"
os.chdir(ROOT)
print(f"Working directory: {Path.cwd()}")
print("Project structure accessible.")

## Create `src/parsers/k66/` Directory and Files

Create the parser package with a dynamic parser implementation. This file will not modify `analyzer.py`.

In [ ]:
k66_dir = ROOT / "src" / "parsers" / "k66"
k66_dir.mkdir(parents=True, exist_ok=True)

k66_init = k66_dir / "__init__.py"

k66_init.write_text(
    '''
"""K66 firmware parser plugin."""

from pathlib import Path
from typing import List, Dict, Any
import re

NAME = "k66"
VERSION = "0.1"
SUPPORTED_FILE_TYPES = [".bin", ".hex"]
STATUS = "active"
DESCRIPTION = "K66 firmware parser for NXP K66 MCU images."

def parse(data: bytes, strings: List[str], path: Path) -> Dict[str, Any]:
    joined = " ".join(strings).lower()
    detected = "k66" in joined or b"k66" in data.lower()
    if not detected:
        return {"detected": False}

    version = None
    m = re.search(r"k66[ _-]?v?([0-9]+(?:\.[0-9]+)*)", joined)
    if m:
        version = m.group(1)

    return {
        "name": NAME,
        "detected": True,
        "firmware_type": "K66",
        "flight_stack": "baremetal",
        "board_type": "K66",
        "processor_family": "ARM Cortex-M4",
        "version": version,
        "confidence": 0.8,
        "notes": "K66 firmware detected",
    }
''' ,
    encoding="utf-8"
 )

print(f"Created parser module at: {k66_init}")

## Implement Parser Metadata and Supported File Types

The parser module declares supported file types and version metadata so it can be picked up by the dynamic discovery system.

In [ ]:
content = k66_init.read_text(encoding="utf-8")
print(content[:400])

## Register Parser via Dynamic Parser System

Load discovered parser modules using the existing `src.parsers` dynamic discovery loader.

In [ ]:
from src.parsers import load_parsers, get_parser_metadata

parsers = load_parsers()
parser_names = [getattr(parser, 'NAME', parser.__name__) for parser in parsers]
print(f"Discovered parsers: {parser_names}")
assert 'k66' in parser_names, 'k66 parser was not discovered dynamically'

k66_module = next(parser for parser in parsers if getattr(parser, 'NAME', None) == 'k66')
print('k66 metadata:', get_parser_metadata(k66_module))

## Verify Parser Appears in Reports

Run the analyzer over sample inputs and inspect generated report and inventory files.

In [ ]:
from pathlib import Path
import subprocess
import json

sample_path = ROOT / "samples" / "input" / "sample_k66_firmware.bin"
sample_path.write_text("K66 firmware image for NXP K66 MCU\n")
print(f"Sample file created: {sample_path}")

result = subprocess.run(["python", "-m", "src.cli", "analyze-all", "--input", "samples/input", "--output", "reports/k66-test"], capture_output=True, text=True)
print("Return code:", result.returncode)
print(result.stdout)
print(result.stderr)
assert result.returncode == 0, 'Analyzer run failed'

parser_inventory_path = ROOT / "reports" / "k66-test" / "parser_inventory.json"
index_html_path = ROOT / "reports" / "k66-test" / "index.html"
assert parser_inventory_path.exists(), f"Parser inventory file not found: {parser_inventory_path}"
assert index_html_path.exists(), f"HTML dashboard not found: {index_html_path}"

inventory = json.loads(parser_inventory_path.read_text(encoding='utf-8'))
print('Parser inventory entries:', [entry['name'] for entry in inventory])
assert any(entry['name'] == 'k66' for entry in inventory), 'k66 missing from parser_inventory.json'

html_content = index_html_path.read_text(encoding='utf-8')
assert 'k66' in html_content.lower(), 'k66 missing from reports/index.html'
print('K66 parser appears in both generated outputs.')